# K-nearest neighbors and SVM classification

Learn two classification methods, keep scaling inside their pipelines, and choose hyperparameters with cross-validation. Iris is bundled with scikit-learn, so this lesson has no external dataset or GPU requirement.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 42
data = load_iris()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target,
    test_size=0.25, stratify=data.target, random_state=SEED)
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
print('Features:', data.feature_names)
print('Classes:', data.target_names.tolist())


## Fit small searches using only the training split

KNN uses distances, so feature scale changes its neighborhood. SVM's `C` controls the penalty on errors; the RBF kernel also uses `gamma` to set a similarity scale. Search scores are validation results, not held-out test results.


In [ ]:
searches = {
    'KNN': GridSearchCV(Pipeline([('scale', StandardScaler()), ('model', KNeighborsClassifier())]),
        {'model__n_neighbors': [3, 5, 9], 'model__weights': ['uniform', 'distance']},
        cv=cv, scoring='accuracy', n_jobs=1),
    'SVM': GridSearchCV(Pipeline([('scale', StandardScaler()), ('model', SVC())]),
        {'model__kernel': ['linear', 'rbf'], 'model__C': [0.1, 1.0, 10.0]},
        cv=cv, scoring='accuracy', n_jobs=1),
}
rows = []
for name, search in searches.items():
    search.fit(X_train, y_train)
    rows.append({'model': name, 'mean_cv_accuracy': search.best_score_, 'settings': search.best_params_})
display(pd.DataFrame(rows))


## Select from cross-validation, then check the untouched test set

This tiny split can give unstable estimates. A high score on 38 Iris examples does not establish performance on a new population.


In [ ]:
chosen_name = max(searches, key=lambda name: searches[name].best_score_)
chosen = searches[chosen_name].best_estimator_
prediction = chosen.predict(X_test)
print('Selected from CV:', chosen_name)
print('Test accuracy:', round(accuracy_score(y_test, prediction), 3))
print(classification_report(y_test, prediction, target_names=data.target_names, zero_division=0))
assert prediction.shape == y_test.shape


## Practice

Add a Manhattan-distance KNN candidate inside the training search. Compare training and validation scores to identify overfitting. Keep the test set for final evaluation.

**References:** [nearest neighbors](https://scikit-learn.org/stable/modules/neighbors.html), [SVM](https://scikit-learn.org/stable/modules/svm.html).
